# Bharat Avatar Platform - Wav2Lip Video Generation API
Run this notebook in Google Colab to provide the Webhook for the backend to generate AI spokesperson videos.

In [ ]:
!git clone https://github.com/Wav2Lip/Wav2Lip.git
!cd Wav2Lip && wget 'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/_layouts/15/download.aspx?share=EdjI7bZlgApMqsKSYpmC8l8BccO1JbSBiI6Z0JqB9e0i4Q' -O checkpoints/wav2lip_gan.pth
!pip install fastapi uvicorn python-multipart nest-asyncio pyngrok librosa==0.9.2


In [ ]:
import os
import uuid
import subprocess
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import FileResponse
import nest_asyncio
from pyngrok import ngrok
import uvicorn

app = FastAPI()

WAV2LIP_DIR = "/content/Wav2Lip"
os.makedirs(f"{WAV2LIP_DIR}/inputs", exist_ok=True)
os.makedirs(f"{WAV2LIP_DIR}/results", exist_ok=True)

@app.get("/")
def health_check():
    return {"status": "Wav2Lip Active"}

@app.post("/generate")
async def generate_video(image: UploadFile = File(...), audio: UploadFile = File(...)):
    req_id = uuid.uuid4().hex[:8]
    img_path = f"{WAV2LIP_DIR}/inputs/{req_id}_img.jpg"
    aud_path = f"{WAV2LIP_DIR}/inputs/{req_id}_aud.mp3"
    
    with open(img_path, "wb") as f: f.write(await image.read())
    with open(aud_path, "wb") as f: f.write(await audio.read())
    
    out_path = f"{WAV2LIP_DIR}/results/{req_id}_out.mp4"
    
    cmd = [
        "python", "inference.py",
        "--checkpoint_path", "checkpoints/wav2lip_gan.pth",
        "--face", img_path,
        "--audio", aud_path,
        "--outfile", out_path,
        "--nosmooth"
    ]
    
    subprocess.run(cmd, cwd=WAV2LIP_DIR)
    return FileResponse(out_path, media_type="video/mp4", filename="output.mp4")

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN" # Replace before running
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url
print("==========================================================")
print(f"Colab Webhook URL: {public_url}")
print("Paste this URL into the .env file as COLAB_URL")
print("==========================================================")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)
